<a href="https://colab.research.google.com/github/ricardoOliveiraN/rna-brasileirao/blob/main/rna_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [10]:
import numpy as np
import matplotlib.pyplot as mat
import pickle as pck
import pandas as pd
from google.colab import drive
import os

# Configurações

In [41]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

#DATASET

### BIBLIOTECA DE ATRIBUTOS:



* gols_mandante: conversão numérica do placar do mandante para cálculos.
* gols_visitante: conversão numérica do placar do visitante para cálculos.
* pontos_mandante: pontuação ganha pelo mandante no jogo (3 para vitória, 1 para empate, 0 para derrota).
* pontos_visitante: pontuação ganha pelo visitante no jogo (3 para vitória, 1 para empate, 0 para derrota).
* mgm_casa: média de gols marcados pelo mandante quando joga dentro de casa.
* mgs_casa: média de gols sofridos (tomados) pelo mandante quando joga dentro de casa
* apr_casa: percentual de aproveitamento de pontos do mandante nos jogos em casa (pontos ganhos / 30 possíveis).
* mgm_fora: média de gols marcados pelo visitante quando joga fora de casa.
* mgs_fora: média de gols sofridos (tomados) pelo visitante quando joga fora de casa.
* apr_fora: percentual de aproveitamento de pontos do visitante nos jogos fora de casa.
* dif_aproveitamento: confronto de regularidade (Aproveitamento do Mandante em Casa menos o Aproveitamento do Visitante Fora).
* dif_ataque: confronto de força ofensiva (Média de gols que o mandante faz em casa menos a média de gols que o visitante sofre fora).
* dif_defesa: confronto de força defensiva (Média de gols que o mandante sofre em casa menos a média de gols que o visitante faz fora).
* resultado: o gabarito que a IA tentará adivinhar (2 para vitória do mandante, 1 para empate, 0 para vitória do visitante).


### Inicia data frame

In [22]:
# conectar com google drive
drive.mount('/content/drive')

# caminho do arquivo
caminho_arquivo = "/content/drive/MyDrive/campeonato-brasileiro-full.csv"

# extrair arquivos do google drive
df = pd.read_csv(caminho_arquivo)

#print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Ajustar tipos e cronologia

In [43]:
df["data"] = pd.to_datetime(
    df["data"],
    format="%d/%m/%Y"
)

df["gols_mandante"] = pd.to_numeric(df["mandante_Placar"])
df["gols_visitante"] = pd.to_numeric(df["visitante_Placar"])

df = (
    df
    .sort_values("data")
    .reset_index(drop=True)
)


### Criar pontos

In [24]:
df["pontos_mandante"] = np.where(
    df["gols_mandante"] > df["gols_visitante"],
    3,
    np.where(
        df["gols_mandante"] == df["gols_visitante"],
        1,
        0
    )
)

df["pontos_visitante"] = np.where(
    df["gols_visitante"] > df["gols_mandante"],
    3,
    np.where(
        df["gols_visitante"] == df["gols_mandante"],
        1,
        0
    )
)


### Histórico mandante

In [25]:
df["mgm_casa"] = (
    df.groupby("mandante")["gols_mandante"]
      .transform(
          lambda x:
          x.shift(1).rolling(10).mean()
      )
)

df["mgs_casa"] = (
    df.groupby("mandante")["gols_visitante"]
      .transform(
          lambda x:
          x.shift(1).rolling(10).mean()
      )
)

df["apr_casa"] = (
    df.groupby("mandante")["pontos_mandante"]
      .transform(
          lambda x:
          x.shift(1).rolling(10).sum() / 30
      )
)

### Histórico visitante

In [26]:
df["mgm_fora"] = (
    df.groupby("visitante")["gols_visitante"]
      .transform(
          lambda x:
          x.shift(1).rolling(10).mean()
      )
)

df["mgs_fora"] = (
    df.groupby("visitante")["gols_mandante"]
      .transform(
          lambda x:
          x.shift(1).rolling(10).mean()
      )
)

df["apr_fora"] = (
    df.groupby("visitante")["pontos_visitante"]
      .transform(
          lambda x:
          x.shift(1).rolling(10).sum() / 30
      )
)

### Features finais

In [27]:
df["dif_aproveitamento"] = (
    df["apr_casa"] -
    df["apr_fora"]
)

df["dif_ataque"] = (
    df["mgm_casa"] -
    df["mgs_fora"]
)

df["dif_defesa"] = (
    df["mgs_casa"] -
    df["mgm_fora"]
)


### Resultado (y)

In [32]:
df["resultado"] = np.where(
    df["gols_mandante"] > df["gols_visitante"],
    2,
    np.where(
        df["gols_mandante"] == df["gols_visitante"],
        1,
        0
    )
)


### Limpando os dados a serem utilizados

In [42]:
colunas_importantes = [
    "data",
    "resultado",
    "mgm_casa", "mgs_casa", "apr_casa",
    "mgm_fora", "mgs_fora", "apr_fora",
    "dif_aproveitamento", "dif_ataque", "dif_defesa"
]

# cria um data frame com apenas as colunas importantes.
df_modelo = df[colunas_importantes].dropna().copy()

# definindo os campos de input do df a ser usado pela IA
x = df_modelo[
    [
        "mgm_casa", "mgs_casa", "apr_casa",
        "mgm_fora", "mgs_fora", "apr_fora",
        "dif_aproveitamento", "dif_ataque", "dif_defesa"
    ]
]

# definindo o output do df a ser usado pela IA
y = df_modelo["resultado"]

# criando as máscaras booleanas baseadas no ano (para filtrar)
filtro_treino = df_modelo["data"].dt.year < 2024
filtro_teste = df_modelo["data"].dt.year == 2024

# separando os dados de Treino (2003-2023)
x_train = x[filtro_treino]
y_train = y[filtro_treino]

# separando os dados de Teste (2024)
x_test = x[filtro_teste]
y_test = y[filtro_teste]

print(f"Valores do x_train: {x_train.head()}")
print(f"Valores do x_test: {x_test.head()}")

print(f"Valores do y_train: {y_train.head()}")
print(f"Valores do y_test: {y_test.head()}")

print(f"Formato do X_train: {x_train.shape}")
print(f"Formato do X_test:  {x_test.shape}")

Valores do x_train:      mgm_casa  mgs_casa  apr_casa  mgm_fora  mgs_fora  apr_fora  dif_aproveitamento  dif_ataque  dif_defesa
225       2.0       0.9  0.566667       1.3       1.9  0.266667            0.300000         0.1        -0.4
226       1.1       0.8  0.466667       1.6       1.4  0.433333            0.033333        -0.3        -0.8
232       1.4       0.5  0.766667       1.2       2.0  0.333333            0.433333        -0.6        -0.7
242       2.2       1.0  0.733333       1.3       2.4  0.133333            0.600000        -0.2        -0.3
244       1.7       1.1  0.633333       1.1       1.8  0.233333            0.400000        -0.1         0.0
Valores do x_test:       mgm_casa  mgs_casa  apr_casa  mgm_fora  mgs_fora  apr_fora  dif_aproveitamento  dif_ataque  dif_defesa
8405       2.0       1.3  0.566667       1.7       2.0  0.300000            0.266667         0.0        -0.4
8406       1.1       1.1  0.400000       0.3       2.4  0.000000            0.400000        -1.